# Detector de partes de botella — pre-etiquetado y entrenamiento

Cuaderno para Google Colab (con GPU gratuita). Entrena el modelo que detecta
`botella`, `tapa`, `etiqueta_frente`, `etiqueta_dorso`, `caja` y `separador`.

**Antes de empezar**: menú *Entorno de ejecución → Cambiar tipo de entorno de
ejecución → GPU (T4)*.

- **Parte A**: pre-etiquetado automático con Grounding DINO → descargás un borrador para corregir en makesense.ai (ver `GUIA_ETIQUETADO.md`).
- **Parte B**: entrenamiento con las etiquetas ya corregidas → descargás `detector_partes.pt`.

In [ ]:
# Verificar que hay GPU asignada
!nvidia-smi -L

## Parte A — Pre-etiquetado automático

Subí el ZIP con los fotogramas extraídos en tu PC con
`python -m contador_botellas.fotogramas` (carpeta `dataset_deteccion\imagenes`).

In [ ]:
from google.colab import files
import zipfile, pathlib

subido = files.upload()  # elegí tu ZIP de imágenes
nombre_zip = next(iter(subido))
pathlib.Path("imagenes").mkdir(exist_ok=True)
with zipfile.ZipFile(nombre_zip) as z:
    z.extractall("imagenes")
total = len(list(pathlib.Path("imagenes").rglob("*.jpg")))
print(f"Imágenes extraídas: {total}")

In [ ]:
# Verificar transformers (ya viene en Colab; el pre-etiquetado usa su
# Grounding DINO oficial — autodistill quedó incompatible y se abandonó)
import transformers
print("transformers", transformers.__version__)

In [ ]:
# Pre-etiquetado con el Grounding DINO oficial de transformers.
# Entiende frases en inglés y marca lo que encuentra. Todas las etiquetas se
# marcan como etiqueta_frente: en makesense se cambia la clase a
# etiqueta_dorso donde corresponda. El separador se marca a mano.
import pathlib
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# Frases en inglés → clase YOLO: 0 botella, 1 tapa, 2 etiqueta_frente, 4 caja
TEXTO = "a wine bottle. a bottle cap. a label on a bottle. an open cardboard box."

def clase_de(frase: str) -> int:
    f = frase.lower()
    if "cap" in f: return 1
    if "label" in f: return 2
    if "box" in f: return 4
    return 0

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
procesador = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
modelo = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny").to(dispositivo)

carpeta = pathlib.Path("borrador"); carpeta.mkdir(exist_ok=True)
imagenes = sorted(pathlib.Path("imagenes").rglob("*.jpg"))
for i, ruta in enumerate(imagenes, 1):
    imagen = Image.open(ruta).convert("RGB")
    entradas = procesador(images=imagen, text=TEXTO, return_tensors="pt").to(dispositivo)
    with torch.no_grad():
        salidas = modelo(**entradas)
    res = procesador.post_process_grounded_object_detection(
        salidas, entradas.input_ids, threshold=0.35, text_threshold=0.30,
        target_sizes=[imagen.size[::-1]])[0]
    ancho, alto = imagen.size
    frases = res.get("text_labels", res.get("labels"))
    lineas = []
    for (x1, y1, x2, y2), frase in zip(res["boxes"].tolist(), frases):
        cx, cy = (x1 + x2) / 2 / ancho, (y1 + y2) / 2 / alto
        an, al = (x2 - x1) / ancho, (y2 - y1) / alto
        lineas.append(f"{clase_de(str(frase))} {cx:.6f} {cy:.6f} {an:.6f} {al:.6f}")
    (carpeta / (ruta.stem + ".txt")).write_text("\n".join(lineas))
    if i % 25 == 0 or i == len(imagenes):
        print(f"{i}/{len(imagenes)} imágenes procesadas")
print("Borrador generado en ./borrador (un .txt por imagen, formato YOLO)")

In [ ]:
# Descargar el borrador para corregirlo en makesense.ai
import shutil
shutil.make_archive("borrador_para_corregir", "zip", "borrador")
files.download("borrador_para_corregir.zip")
print("Seguí en GUIA_ETIQUETADO.md, paso 2 (makesense.ai). Después volvé a la Parte B.")

## Parte B — Entrenamiento con las etiquetas corregidas

Subí **un solo ZIP** con esta estructura (imágenes + los `.txt` exportados de
makesense en formato YOLO, con el mismo nombre que cada imagen):

```
dataset.zip
├── imagenes/   (todas las .jpg)
└── etiquetas/  (todos los .txt de makesense)
```

In [ ]:
from google.colab import files
import zipfile, pathlib

subido = files.upload()  # elegí dataset.zip
with zipfile.ZipFile(next(iter(subido))) as z:
    z.extractall("corregido")
imagenes = sorted(pathlib.Path("corregido").rglob("*.jpg"))
etiquetas = sorted(pathlib.Path("corregido").rglob("*.txt"))
print(f"{len(imagenes)} imágenes, {len(etiquetas)} archivos de etiquetas")

In [ ]:
# Armar la estructura train/val que espera YOLO (85% / 15%, reproducible)
import random, shutil, pathlib

CLASES = ["botella", "tapa", "etiqueta_frente", "etiqueta_dorso", "caja", "separador"]
rng = random.Random(37)

pares = []
mapa_txt = {ruta.stem: ruta for ruta in pathlib.Path("corregido").rglob("*.txt")}
for imagen in pathlib.Path("corregido").rglob("*.jpg"):
    txt = mapa_txt.get(imagen.stem)
    if txt is not None:
        pares.append((imagen, txt))
print(f"Pares imagen+etiqueta: {len(pares)}")

rng.shuffle(pares)
corte = max(1, int(len(pares) * 0.15))
particiones = {"val": pares[:corte], "train": pares[corte:]}
raiz = pathlib.Path("dataset_yolo")
if raiz.exists():
    shutil.rmtree(raiz)
for nombre, lote in particiones.items():
    for imagen, txt in lote:
        destino_img = raiz / "images" / nombre
        destino_lbl = raiz / "labels" / nombre
        destino_img.mkdir(parents=True, exist_ok=True)
        destino_lbl.mkdir(parents=True, exist_ok=True)
        shutil.copy2(imagen, destino_img / imagen.name)
        shutil.copy2(txt, destino_lbl / (imagen.stem + ".txt"))

contenido = ["path: " + str(raiz.resolve()), "train: images/train", "val: images/val",
             "names:"] + [f"  {i}: {c}" for i, c in enumerate(CLASES)]
(raiz / "data.yaml").write_text("\n".join(contenido))
print((raiz / "data.yaml").read_text())

In [ ]:
# Entrenar (20-60 min en la T4 gratuita; corta solo antes si deja de mejorar)
%pip install -q "ultralytics==8.4.92"
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")
modelo.train(data="dataset_yolo/data.yaml", epochs=100, imgsz=640, patience=25)
print("Mejor modelo:", modelo.trainer.best)

In [ ]:
# Ver el rendimiento por clase (mAP): si una clase anda mal, hacen falta más muestras de esa clase
metricas = modelo.val(data="dataset_yolo/data.yaml")

In [ ]:
# Descargar el modelo final: copiarlo a C:\contador\modelos\ en la PC de la planta
import shutil
from google.colab import files

shutil.copy2(modelo.trainer.best, "detector_partes.pt")
files.download("detector_partes.pt")
print("Probalo con: python -m contador_botellas --fuente 0 --modelo modelos\\detector_partes.pt --tablero")